In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
cancer = pd.read_csv('breast-cancer.csv')

In [ ]:
cancer.info()

In [ ]:
cancer = cancer.drop('id', axis=1)

In [ ]:
cancer['diagnosis'].unique()

In [ ]:
cancer['diagnosis'] = cancer['diagnosis'].map({'M':0, 'B': 1})

## **MODEL**

In [ ]:
X = cancer.drop('diagnosis', axis=1)
y = cancer['diagnosis']

In [ ]:
# # do we need scaling tho?
# # Normalisasi
# X = (X - X.min()) / (X.max() - X.min())

# # Standarisasi
# X = (X - X.mean()) / X - X.std()

In [ ]:
np.random.seed(42)

idx_0 = np.where(y == 0)[0]
idx_1 = np.where(y == 1)[0]

np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

train_0 = int(len(idx_0) * 0.8)
train_1 = int(len(idx_1) * 0.8)

train_idx = np.concatenate([idx_0[:train_0], idx_1[:train_1]])
test_idx = np.concatenate([idx_0[train_0:], idx_1[train_1:]])

np.random.shuffle(train_idx)
np.random.shuffle(test_idx)

X_train = X.iloc[train_idx].values
y_train = y.iloc[train_idx].values
X_test = X.iloc[test_idx].values
y_test = y.iloc[test_idx].values

In [ ]:
def linear(X, w,b):
    z = np.dot(X, w) + b
    return z

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
def loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    loss = -np.mean(y_true  * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss

In [ ]:
def gradient(X, y_true, y_pred):
    m = X.shape[0]
    error = y_pred - y_true
    dw = (1 / m) * np.dot(X.T, error)
    db = (1 / m) * np.sum(error)
    return dw, db

In [ ]:
def update_parameter(w, b, dw, db, learning_rate):
    w_new = w - learning_rate * dw
    b_new = b - learning_rate * db

    return w_new, b_new

In [ ]:
X_train.shape[1]

In [ ]:
w = np.zeros(X_train.shape[1])
b = 0 
learning_rate = 15
ephocs = 700
loss_history = []

for i in range(ephocs):
    z = linear(X_train, w, b)
    y_pred = sigmoid(z)

    loss = loss(y_train, y_pred)
    loss_history.append(loss)

    dw, db = gradient(X_train, y_train, y_pred)

    w, b = update_parameter(w, b, dw, db, learning_rate)

    if i % 50 == 0:
        print(f"epoch : {i}, loss : {loss}")

In [ ]:
def prediksi(X, w, b, threshold=0.7):
    z = linear(X, w, b)
    y_prob = sigmoid(z)

    y_class = [1 if p > threshold else 0 for p in y_prob]

    return np.array(y_class)

## **EVALUASI**

In [ ]:
def evaluate(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) /  (precision + recall) if (precision + recall) > 0 else 0

    return accuracy, precision, recall, f1

In [ ]:
prediksi_train = prediksi(X_train, w, b)
akurasi_train = np.sum(prediksi_train == y_train) / len(X_train)
print(f"Akurasi train : {akurasi_train}")

In [ ]:
prediksi_test = prediksi(X_test, w, b)
akurasi_test = np.sum(prediksi_test == y_test) / len(X_test)
print(f"Akurasi test : {akurasi_test}")

In [ ]:
_, precision_train, recall_train, f1_train = evaluate(y_train, prediksi_train)
_, precision_test, recall_test, f1_test = evaluate(y_test, prediksi_test)

print(f"Train\nPresisi: {precision_train}\nRecall: {recall_train}\nF1: {f1_train}")
print(f"\n\nTest\nPresisi: {precision_test}\nRecall: {recall_test}\nF1: {f1_test}")